<a href="https://colab.research.google.com/github/suder54ul/LLMP/blob/main/module03_handson1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 03 - Hands-on 1: Building Reliable Prompt Engineering Pipelines
**Large Language Models in Production**
**Open – May 2026**

**Facilitator:** A/P TAN Wee Kek  
**Email:** distwk@nus.edu.sg

**Learning Objectives**
By the end of this exercise, you will be able to:
- Understand why raw/naïve prompts are unreliable in production
- Build a structured **prompt engineering pipeline** with all 7 components from the lecture
- Create reusable `ChatPromptTemplate` objects in LangChain
- Compare output quality, consistency, and production-readiness before vs. after

**Stack:** Ollama + Meta LLaMA 3.1 8B Instruct + LangChain

In [ ]:
# Setup - Run this cell first
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Use the exact model from the course
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.0,      # deterministic for fair comparison
    # temperature=0.7,      # more creative for better demonstration of prompt engineering
    num_ctx=8192,
)

print("LLaMA 3.1 8B Instruct is ready via Ollama!")

## Part 1: Why Simple (Naïve) Prompts Fail in Production

A typical "demo-style" prompt often used in early experiments.

In [ ]:
simple_prompt = "Summarise the following document for a senior manager."

document = """Q2 2026 Financial Performance Report
Revenue grew 12% YoY to SGD 245 million, driven by strong demand in the APAC region.
However, supply chain disruptions in semiconductor components increased costs by 8%.
Regulatory changes in data privacy (PDPA amendments) require immediate compliance updates.
Customer churn increased slightly in the enterprise segment due to competitive pricing."""

# Run the simple prompt
response_simple = llm.invoke(simple_prompt + "\n\nDocument:\n" + document)
print("=== SIMPLE PROMPT OUTPUT ===\n")
print(response_simple.content)

## Part 2: Structured Prompt Pipeline (Manual Version)

Now apply the full **7-component prompt engineering pipeline** from the lecture.

In [ ]:
structured_system_prompt = """You are a senior strategy consultant for a Singapore-based enterprise.
You provide clear, actionable insights to C-suite executives."""

structured_user_prompt = f"""### Context
{document}

### Task
Summarise the document for a senior manager.

### Constraints
- Focus only on business implications, risks, and recommended next steps
- Maximum 5 bullet points
- If any information is missing, explicitly state \"Information not available\"

### Output Format
Return ONLY the bullet points. No introduction or conclusion text.

### Safety & Compliance Instructions
- Ensure PDPA compliance. Do not disclose any personal data.
- Clearly flag any regulatory or compliance risks.
- Never suggest actions that could violate Singapore laws or regulations."""

messages = [
    ("system", structured_system_prompt),
    ("human", structured_user_prompt)
]

response_structured = llm.invoke(messages)
print("=== STRUCTURED PIPELINE OUTPUT ===\n")
print(response_structured.content)

## Part 3: Production-Grade Reusable PromptTemplate with LangChain

Create a reusable template that can be applied to many different documents and tasks.

In [ ]:
# Define a reusable ChatPromptTemplate with all 7 components
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are {role}.
You are helping a Singapore-based enterprise make strategic decisions.
Always follow PDPA, company governance rules, and best practices for responsible AI."""),

    ("human", """### Context
{context}

### User Instruction
{user_instruction}

### Constraints
{constraints}

### Output Format
{output_format}

### Safety Instructions
{safety_instructions}""")
])

# Create the LCEL chain
chain = prompt_template | llm | StrOutputParser()

# Example usage with the same document
result = chain.invoke({
    "role": "senior strategy consultant",
    "context": document,
    "user_instruction": "Summarise the document for a senior manager. Focus on business implications, risks and recommended next steps.",
    "constraints": "Maximum 5 bullet points. If evidence is missing, state explicitly.",
    "output_format": "Return ONLY the bullet points. No introduction or conclusion.",
    "safety_instructions": "Ensure PDPA compliance. Flag any regulatory risks. Never suggest actions that could violate Singapore regulations."
})

print("=== LANGCHAIN PROMPT TEMPLATE OUTPUT ===\n")
print(result)

## Part 4: Your Turn – Build Your Own Production Pipeline (15–20 minutes)

**Choose ONE** of the following real-world scenarios (or bring your own company use case):

1. Draft a professional email response to a customer complaint
2. Extract key action items and risks from a policy / meeting document
3. Generate a compliance checklist from new regulatory text
4. Create an executive summary from a quarterly business review

**Instructions:**
1. Define values for **all 7 components** (role, context, instruction, constraints, output format, safety, tone)
2. Modify the `prompt_template` above or create a new one
3. Run the chain with your new input
4. (Optional) Change `temperature=0.7` and run the same input 3 times to observe consistency

**Tip:** Copy the template cell and experiment!

In [ ]:
# Enter your own custom document and instructions here to test the reusable prompt template
# Start with temperature=0.0 for deterministic output, then experiment with higher temperatures of 0.7 for more creative responses.



## Part 5: Group Discussion & Production Implications (10 minutes)

Discuss with your neighbour:

1. How much did the structured pipeline improve output quality and reliability?
2. Which of the 7 components had the biggest impact in your experiment?
3. What production risks are reduced by using well-designed prompt pipelines?
4. How would you version-control, A/B test, and maintain these templates in a real enterprise system?

**Key Takeaway (from lecture slide):**  
**A production LLM system is only as good as its prompt engineering pipeline.**

Well-structured prompts are the foundation for RAG, Tool Use, Agents, and everything else we will cover in this course.